# Exploratory Data Analysis (EDA)
## AI-Based Student Performance Risk Predictor

This notebook examines the supplied 15,000-row student dataset before model training.

**Questions we will investigate**
1. What are the columns, data types, ranges and class counts?
2. Are there missing values, duplicate records or suspicious values?
3. How are numeric features distributed?
4. How do feature distributions differ across Low, Medium and High performance?
5. Which features are associated with the target label?
6. Is there evidence that the target label may have been constructed from the input features?

> Important: EDA can reveal patterns and potential leakage, but it cannot prove how the labels were generated. The dataset's documentation or generation process is needed to establish that.


In [ ]:
# If needed, install packages in your environment:
# %pip install pandas numpy matplotlib seaborn

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display as ipy_display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Keep the CSV in the same folder as this notebook, or change this path.

# Keep the CSV in the same folder as this notebook
DATASET_NAME = "month_3_ann_student_performance_dataset_15000 (1).csv"

DATA_PATH = Path.cwd() / DATASET_NAME

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Please place the CSV file in the project root directory."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
ipy_display(df.head())



## 1. Structure and data types

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "unique_values": df.nunique()
}))


## 2. Data quality checks

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate student_id values:", df["student_id"].duplicated().sum())

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
display(df[numeric_cols].describe().T)

# Check values against plausible domain bounds. These are review flags, not automatic deletion rules.
bounds = {
    "age": (0, 100),
    "attendance_pct": (0, 100),
    "study_hours_per_week": (0, 168),
    "previous_score": (0, 100),
    "assignment_completion_pct": (0, 100),
    "internal_assessment_score": (0, 100),
    "participation_score": (0, 10),
    "sleep_hours_per_day": (0, 24),
    "extracurricular_hours_per_week": (0, 168),
    "failed_subjects": (0, None),
}
for col, (low, high) in bounds.items():
    invalid = pd.Series(False, index=df.index)
    if low is not None:
        invalid |= df[col] < low
    if high is not None:
        invalid |= df[col] > high
    print(f"{col}: {invalid.sum()} values outside review bounds")


## 3. Target variable distribution

Check whether the Low, Medium and High classes are balanced. A balanced target is useful, but does not by itself guarantee a valid or generalizable model.

In [ ]:
target = "performance_category"
print(df[target].value_counts(dropna=False))
print("\nPercentages:")
display((df[target].value_counts(normalize=True, dropna=False) * 100).round(2))

plt.figure(figsize=(7, 4))
order = df[target].value_counts().index
sns.countplot(data=df, x=target, order=order)
plt.title("Student count by performance category")
plt.xlabel("Performance category")
plt.ylabel("Number of students")
plt.tight_layout()
plt.show()


## 4. Distributions of numerical features

In [ ]:
feature_cols = [
    c for c in df.select_dtypes(include=np.number).columns
    if c != "student_id"
]

ncols = 3
nrows = int(np.ceil(len(feature_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, feature_cols):
    sns.histplot(data=df, x=col, kde=True, ax=ax)
    ax.set_title(f"Distribution: {col}")

for ax in axes[len(feature_cols):]:
    ax.remove()

plt.tight_layout()
plt.show()


**How to interpret:** Look for skewed distributions, unusual spikes, impossible values and features with very different scales. Do not remove outliers automatically; first decide whether they are errors or legitimate observations.

## 5. Feature distributions by performance category

In [ ]:
# Boxplots help compare the median, spread and potential outliers by class.
ncols = 2
nrows = int(np.ceil(len(feature_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
axes = np.array(axes).reshape(-1)

class_order = ["Low", "Medium", "High"]
for ax, col in zip(axes, feature_cols):
    sns.boxplot(data=df, x=target, y=col, order=class_order, ax=ax)
    ax.set_title(f"{col} by performance category")

for ax in axes[len(feature_cols):]:
    ax.remove()

plt.tight_layout()
plt.show()


**How to interpret:** If a feature's values are clearly separated across classes, it may be strongly associated with the target. Overlap between boxes means the feature alone does not cleanly distinguish the classes. Association is not causation.

## 6. Average feature values by category

In [ ]:
class_means = df.groupby(target)[feature_cols].mean().reindex(class_order)
display(class_means.round(2))

class_means.T.plot(kind="bar", figsize=(15, 6))
plt.title("Mean feature values by performance category")
plt.xlabel("Input feature")
plt.ylabel("Mean value")
plt.xticks(rotation=55, ha="right")
plt.legend(title="Performance category")
plt.tight_layout()
plt.show()


## 7. Correlation between numerical features

In [ ]:
corr = df[feature_cols].corr(numeric_only=True)

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation heatmap of numerical input features")
plt.tight_layout()
plt.show()

# Show the strongest absolute feature-feature correlations, excluding self-correlations.
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
strong_pairs = upper.stack().rename("correlation").reset_index()
strong_pairs.columns = ["feature_1", "feature_2", "correlation"]
display(strong_pairs.reindex(strong_pairs["correlation"].abs().sort_values(ascending=False).index).head(15))


## 8. Target-label relationship checks

The target is categorical, so a numeric correlation with the target is not automatically meaningful. First compare class-wise feature distributions and use simple statistical summaries.

The next cell also checks whether any individual feature almost perfectly separates a class. This is a warning signal to investigate possible rule-based label construction or leakage, not proof by itself.

In [ ]:
# Class-wise means and standard deviations
summary = df.groupby(target)[feature_cols].agg(["mean", "std", "min", "median", "max"])
display(summary.round(2))

# Quantiles by class for key academic features
key_features = [
    "study_hours_per_week",
    "previous_score",
    "internal_assessment_score",
    "attendance_pct",
    "assignment_completion_pct",
    "failed_subjects",
]
for col in key_features:
    print(f"\\n{col}")
    display(df.groupby(target)[col].describe()[["min", "25%", "50%", "75%", "max"]].round(2))


## 9. Check for possible target construction or leakage

A synthetic dataset may assign Low/Medium/High labels using a score or thresholds derived from the same columns supplied as inputs. If so, a model can learn those rules and obtain high test accuracy without demonstrating real-world prediction of future performance.

Inspect:
- Whether one feature nearly determines the class.
- Whether class boundaries appear unusually sharp.
- Whether the dataset has a data-generation description.
- Whether any input feature was calculated using the target or future information.

Do not remove features solely because they are correlated with the target: strong predictors are expected. Establish how the target was created before deciding what is leakage.

In [ ]:
# For each numeric feature, show class-wise ranges and overlap.
for col in feature_cols:
    print(f"\\n{col}")
    display(df.groupby(target)[col].agg(["min", "max", "mean", "median"]).round(2))

# Optional: inspect a sample of records around category boundaries manually.
display(df.sample(min(12, len(df)), random_state=42).sort_values(target))


## 10. EDA conclusions — complete after reviewing the plots

Fill in these points based on the actual charts and tables:

1. **Data quality:** Are missing values, duplicates or invalid values present?
2. **Class balance:** Are Low, Medium and High reasonably balanced?
3. **Feature distributions:** Which features are skewed or contain unusual values?
4. **Class separation:** Which features differ most across the three classes?
5. **Feature relationships:** Are any input features strongly correlated with each other?
6. **Potential leakage:** Is there evidence that the target may have been generated from the input features?
7. **Preprocessing decisions:** Which columns should be excluded, encoded or scaled?

### Important modelling note
- Exclude `student_id` from model inputs.
- Use a stratified train/test split after deciding on preprocessing.
- Fit scalers on the training data only, then apply them to validation/test data.
- Keep the test set untouched until final evaluation.
- Compare ANN results with the required Logistic Regression baseline.


# 11. Data Preprocessing

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. Separate input features and target
X = df.drop(columns=["student_id", "performance_category"])
y = df["performance_category"]

# 2. Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Target classes:", label_encoder.classes_)

# 3. Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

# 4. Feature scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Preprocessing completed successfully!")


In [ ]:
print("Actual training feature order:")
print(list(X.columns))

# 12. Baseline Model - Logistic Regression

In [ ]:

# Import required libraries
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# 1. Create Logistic Regression model
log_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# 2. Train the model
log_model.fit(X_train_scaled, y_train)

print("Logistic Regression training completed!")

# 3. Make predictions
y_pred_log = log_model.predict(X_test_scaled)

# 4. Calculate accuracy
accuracy_log = accuracy_score(y_test, y_pred_log)

print("\nLogistic Regression Accuracy:",
      round(accuracy_log * 100, 2), "%")

# 5. Classification report
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_log,
    target_names=label_encoder.classes_
))

# 6. Confusion matrix
cm = confusion_matrix(y_test, y_pred_log)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_
)

display.plot(cmap="Blues")
plt.title("Logistic Regression - Confusion Matrix")
plt.show()


# 13. Artificial Neural Network (ANN) Model

In [ ]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

# Set random seed
tf.keras.utils.set_random_seed(42)

# Create ANN architecture
ann_model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),

    Dense(64, activation="relu"),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

# Compile model
ann_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Display architecture
ann_model.summary()

# Early stopping
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

# Train model
history = ann_model.fit(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.20,
    callbacks=[early_stop],
    verbose=1
)

print("ANN training completed successfully!")


# 14. ANN Training Performance Visualization

In [ ]:

import matplotlib.pyplot as plt

# Plot training and validation accuracy
plt.figure(figsize=(9, 5))

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.title("ANN Training vs Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.show()


In [ ]:

# Plot training and validation loss
plt.figure(figsize=(9, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("ANN Training vs Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.show()


# 15. ANN Model Evaluation

In [ ]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# Evaluate ANN
test_loss, test_accuracy = ann_model.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print("ANN Test Loss:", round(test_loss, 4))
print("ANN Test Accuracy:", round(test_accuracy * 100, 2), "%")

# Generate predictions
y_pred_prob = ann_model.predict(X_test_scaled)

# Convert probabilities into class labels
y_pred_ann = y_pred_prob.argmax(axis=1)

# Classification report
print("\nANN Classification Report:")
print(classification_report(
    y_test,
    y_pred_ann,
    target_names=label_encoder.classes_
))

# Confusion matrix
cm_ann = confusion_matrix(y_test, y_pred_ann)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_ann,
    display_labels=label_encoder.classes_
)

disp.plot(cmap="Blues")
plt.title("ANN Confusion Matrix")
plt.show()


In [ ]:

# Compare both models
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Artificial Neural Network"],
    "Accuracy (%)": [
        accuracy_log * 100,
        test_accuracy * 100
    ]
})

# Display comparison without using the overwritten display()
print(comparison.round(2).to_string(index=False))

# Plot comparison
plt.figure(figsize=(8, 5))

plt.bar(
    comparison["Model"],
    comparison["Accuracy (%)"]
)

plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy (%)")
plt.ylim(0, 100)
plt.xticks(rotation=10)
plt.grid(axis="y")

plt.tight_layout()
plt.show()


# 16. Saving the Trained ANN Model

In [ ]:

import os
import joblib

# Create models directory
os.makedirs("models", exist_ok=True)

# Save ANN model
ann_model.save("models/student_performance.keras")

# Save scaler
joblib.dump(
    scaler,
    "models/scaler.pkl"
)

# Save label encoder
joblib.dump(
    label_encoder,
    "models/label_encoder.pkl"
)

print("All model files saved successfully!")


In [ ]:
import os

print(os.listdir("models"))